# GRPO on a verifiable architecture-design environment

Train a small open model (default: Qwen2.5-1.5B-Instruct) to design neural-network architectures, with rewards computed by the deterministic verifier from [neurarch-arch-bench](https://github.com/neurarch-ai/neurarch-arch-bench).

No human labels, no LLM judge: a task is a natural-language spec plus a starting graph, the policy emits graph-edit actions, and a sub-millisecond programmatic verifier grades the result (structural blockers, budget, connectivity, required layer types).

Runtime: a T4 (Colab free tier) works with `--lora`; an A100 is comfortable. The env server is plain Node with zero dependencies.

Plan: (1) start the env server, (2) measure baseline pass@1 on a held-out split, (3) GRPO-train on a training split, (4) re-measure on the held-out split, (5) plot the reward curve.

In [ ]:
# 1. Get the environment and start the reward server.
# PUBLIC repo: this clones it. PRIVATE repo: upload neurarch-arch-bench.zip via
# the Colab Files panel (left sidebar) FIRST, then run this cell to unzip it.
import os
if not os.path.isdir('neurarch-arch-bench'):
    if os.path.exists('neurarch-arch-bench.zip'):
        !unzip -q neurarch-arch-bench.zip
    else:
        !git clone --depth 1 https://github.com/neurarch-ai/neurarch-arch-bench
%cd neurarch-arch-bench
import subprocess, time
subprocess.Popen(['node', 'env-server.mjs'])
time.sleep(2)
import urllib.request, json as _j
print(_j.load(urllib.request.urlopen('http://localhost:8737/health')))


In [ ]:
# 2. Python deps.
%pip install -q "trl>=0.14" transformers datasets accelerate peft bitsandbytes
# Colab ships an old torchao that PEFT's LoRA path rejects (raises instead of
# skipping). LoRA does not need it, so remove it to avoid an ImportError.
%pip uninstall -y torchao


In [ ]:
# 3. Baseline: pass@1 of the untrained model on a HELD-OUT split (seed 999).
#    Keep this number; it is the honest "before".
!python training/train_grpo.py --eval-only --seed 999 --count 64

In [ ]:
# 4. GRPO training on the seed-123 split (disjoint from the eval split).
#    ~300 steps x 8 generations is a few hours on a T4; shrink --steps for a
#    quick signal check.
!python training/train_grpo.py --steps 300 --count 512 --seed 123 --lora --bf16

In [ ]:
# 5. Reward curve: mean sampled reward per step. Rising = the verifier's
#    signal is learnable.
import json
import matplotlib.pyplot as plt

state = json.load(open('out/grpo-arch/trainer_state.json'))
steps = [h['step'] for h in state['log_history'] if 'reward' in h]
rewards = [h['reward'] for h in state['log_history'] if 'reward' in h]
plt.plot(steps, rewards)
plt.xlabel('step'); plt.ylabel('mean reward'); plt.title('GRPO on neurarch-arch-bench')
plt.savefig('reward_curve.png', dpi=150)
plt.show()

In [ ]:
# 6. After: same held-out split, trained checkpoint. Compare pass@1 and parse
#    failures against step 3.
!python training/train_grpo.py --eval-only --seed 999 --count 64 --model out/grpo-arch/checkpoint-final

## Reading the result

Three numbers matter, all on the held-out split (seed 999, never trained on):

1. **pass@1 before vs after**: did verifiable-reward training generalize to unseen tasks?
2. **parse failures before vs after**: the first thing GRPO usually fixes is emitting valid JSON action plans.
3. **the reward curve**: rising mean reward on the training split shows the signal is learnable at all.

Because the split generator is seeded and deterministic, anyone can reproduce this exact experiment: same seeds, same tasks, same verifier.